# task_15 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

listings = pd.read_csv(base / "listings.csv")
orders = pd.read_csv(base / "orders.csv")
refunds = pd.read_csv(base / "refunds.csv")
reviews = pd.read_csv(base / "reviews.csv")

orders["delivered_date"] = pd.to_datetime(orders["delivered_date"])
refunds["refund_date"] = pd.to_datetime(refunds["refund_date"])

ordx = orders.merge(listings[["listing_id", "seller_id", "category", "price"]], on="listing_id")
refund_by_order = refunds.groupby("order_id")["refund_amount"].sum().rename("refund_amount")
ordx = ordx.join(refund_by_order, on="order_id").fillna({"refund_amount": 0})
ordx["net_revenue"] = ordx["price"] * ordx["quantity"] - ordx["refund_amount"]

seller_net = ordx.groupby("seller_id")["net_revenue"].sum()
active_listings = ordx.groupby("seller_id")["listing_id"].nunique()
q1 = str((seller_net / active_listings).sort_values(ascending=False).index[0])

ratings = reviews.merge(orders[["order_id", "listing_id"]], on="order_id").merge(listings[["listing_id", "category"]], on="listing_id")
q2 = str(ratings.groupby("category")["rating"].mean().sort_values(ascending=True).index[0])

refund_timing = refunds.merge(orders[["order_id", "delivered_date"]], on="order_id")
refund_timing["days_after_delivery"] = (refund_timing["refund_date"] - refund_timing["delivered_date"]).dt.days
refunded_within_7 = set(refund_timing[refund_timing["days_after_delivery"] <= 7]["order_id"])
q3 = f"{(100 * len(refunded_within_7) / orders['order_id'].nunique()):.1f}%"

refunded_orders = set(refunds["order_id"])
total_by_region = orders.groupby("buyer_region")["order_id"].nunique()
refunded_by_region = orders[orders["order_id"].isin(refunded_orders)].groupby("buyer_region")["order_id"].nunique()
q4 = str((refunded_by_region / total_by_region).sort_values(ascending=False).index[0])

per_listing = ordx.groupby("listing_id").agg(net=("net_revenue", "sum"), num_orders=("order_id", "nunique"))
per_listing["net_per_order"] = per_listing["net"] / per_listing["num_orders"]
q5 = str(per_listing["net_per_order"].sort_values(ascending=False).index[0])


Q1: Which seller_id has the highest net revenue per active listing, where net revenue = pricequantity - refund_amount and active listings are listings with at least one order?

In [ ]:
print(q1)


Q2: Which category has the lowest average review rating?

In [ ]:
print(q2)


Q3: What percentage of all orders were refunded within 7 days of delivered_date, rounded to 1 decimal place?

In [ ]:
print(q3)


Q4: Which buyer_region has the highest refund rate measured as refunded orders divided by all orders from that region?

In [ ]:
print(q4)


Q5: Which listing_id has the highest net revenue per order?

In [ ]:
print(q5)
